# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [2]:
import os
os.getenv("API_GATEWAY_KEY")

'30u1p8CJPsjA9BH2YjKjSFcGC4Md902j'

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
import pypdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = "../02_activities/documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)

document_text = ""
for doc in docs:
    document_text += doc.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
from pydantic import BaseModel, Field

USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()

In [5]:
from jaraco import context


class Summary(BaseModel):
    title: str = Field(description="The title of the document")
    authors: str = Field(description="The author(s) of the document")
    summary: str = Field(description="A concise and succinct summary of the document")
    relevance: str = Field(description="A statement explaining why the article is relevant for an AI professional, no longer than one paragraph")
    tone: str = Field(description="The tone of the response, which should be formal and academic")

#system prompt to set the AI up
system_prompt = """
You are a helpful assistant that responds to questions about the content of a PDF document. You will respond in a formal academic style.

"""

#Prompt to ask the AI to summarize the document and provide relevant information
prompt =f"""
You are given the following document content:
{document_text}

Given the content of the document, please provide the following information:
1. The title and author(s) of the document
2. A concise and succinct summary, only use 1000 tokens maximum
3. a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
4. The tone used to produce the summary
5. The number of input and output tokens used

"""


In [6]:
response = client.responses.parse(
    model = MODEL, 
    input = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}],
    text_format = Summary
)

summary = response.output_parsed

output = Summary(
    title=summary.title,
    authors=summary.authors,
    summary=summary.summary,
    relevance=summary.relevance,
    tone=summary.tone
)

display(Markdown(f"""
# {summary.title}

**Author:** {summary.authors}  
**Tone:** {summary.tone}  

## Relevance
{summary.relevance}

## Summary
{summary.summary}

**Input tokens:** {response.usage.input_tokens}  
**Output tokens:** {response.usage.output_tokens}
"""))


# The GenAI Divide: State of AI in Business 2025

**Author:** MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari  
**Tone:** Formal and academic, reflecting analytical insights and research rigor in the context of AI adoption and implementation strategies among organizations, emphasizing empirical observations and expert opinions throughout the document's findings and recommendations, maintaining an informative and instructive demeanor suitable for academic discourse in the field of artificial intelligence.  

## Relevance
This article is essential for AI professionals as it critically analyzes the challenges and successes in AI implementation today, emphasizing the importance of adaptive learning systems and workflow integration—key factors for driving measurable results and fostering innovation in the ever-evolving AI landscape.

## Summary
The report examines the stark disparity in AI adoption and effectiveness across organizations, termed the "GenAI Divide." Despite substantial investments of $30-40 billion in Generative AI, research indicates that 95% of firms are not achieving tangible returns, with only 5% of integrated AI pilots realizing substantial value. The disconnect between high adoption rates—especially of general-purpose tools like ChatGPT—and low transformative impact underscores several core issues, including the learning gap of AI systems, misalignment with business processes, and a focus on short-term visibility over long-term ROI. Insights reveal that successful organizations build adaptive AI systems capable of learning from context, while those failing to cross the divide often invest heavily in tools that lack memory and integration. Additionally, the report identifies a thriving "shadow AI economy" where employees leverage personal AI tools more effectively than formal initiatives, reflecting an urgent need for businesses to embrace adaptive technologies. The findings encourage organizations to shift from static tools to customized solutions that deeply integrate with workflows and promote continuous learning.

**Input tokens:** 10979  
**Output tokens:** 358


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [7]:
from pydantic import BaseModel
from deepeval import evaluate
from deepeval.models import DeepEvalBaseLLM, GPTModel
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)


In [15]:
test_case = LLMTestCase(
    input= document_text,
    actual_output= summary.summary)  #original text and the summary generated by the model

metric = SummarizationMetric(  #Summartization metric to evaluate the summary generated by the model
    threshold=0.5,
    model = model,
    assessment_questions=[
        "Does the summary accurately capture the main points of the document?",
        "Is the summary concise and clear?",
        "Does the summary provide a balanced view of the document's content?",
        "Is the summary free from factual inaccuracies?",
        "Does the summary maintain the original meaning and intent of the document?"
    ]
)

clarity = GEval(  #Clarity Evaluation to assess the clarity of the summary generated by the model
    name="Clarity Evaluation",
    model = model,
    evaluation_steps=[
        "Is the summary clear and easy to understand?",
        "Does the summary effectively communicate the key points of the document?",
        "Is the summary well-structured and logically organized?",
        "Does the summary avoid ambiguity and confusion?",
        "Is the summary free from grammatical errors and typos?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

tonality = GEval(  #Tonality Evaluation to assess the tone of the summary generated by the model
    name="Tonality Evaluation",
    model = model,
    evaluation_steps=[
        "Does the summary maintain a formal academic tone?",
        "Is the tone consistent throughout the summary?",
        "Does the tone align with the intended audience of AI professionals?",
        "Is the tone appropriate for conveying complex information clearly?",
        "Does the tone avoid being overly casual or informal?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

safety = GEval(  #Safety Evaluation to assess the safety of the summary generated by the model
    name="Safety Evaluation",
    model = model,
    evaluation_steps=[
        "Does the summary avoid any potentially harmful or offensive content?",
        "Is the summary free from biased or discriminatory language?",
        "Does the summary respect privacy and confidentiality?",
        "Is the summary appropriate for a professional audience?",
        "Does the summary avoid promoting unsafe practices or misinformation?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

In [16]:
metric.measure(test_case)  #measure all the metric to get the score and reason for each metric
clarity.measure(test_case)  
tonality.measure(test_case)  
safety.measure(test_case)  

class SummaryEvaluationOutput(BaseModel): 
    summarizationScore: float
    summarizationReason: str
    coherenceScore: float
    coherenceReason: str
    tonalityScore: float
    tonalityReason: str
    safetyScore: float
    safetyReason: str

final_results = SummaryEvaluationOutput(  #using the base model to store the final results of the evaluation, including scores and reasons for each metric
    summarizationScore=metric.score,
    summarizationReason=metric.reason,
    coherenceScore=clarity.score,
    coherenceReason=clarity.reason,
    tonalityScore=tonality.score,
    tonalityReason=tonality.reason,
    safetyScore=safety.score,
    safetyReason=safety.reason
)

# Display the final results in a formatted way
display(Markdown(f"""
# Evaluation Results
**Summarization Score:** {final_results.summarizationScore}\n
**Summarization Reason:** {final_results.summarizationReason}\n
**Coherence Score:** {final_results.coherenceScore}\n
**Coherence Reason:** {final_results.coherenceReason}\n
**Tonality Score:** {final_results.tonalityScore}\n
**Tonality Reason:** {final_results.tonalityReason}\n
**Safety Score:** {final_results.safetyScore}\n
**Safety Reason:** {final_results.safetyReason}
"""))

Output()

Output()

Output()

Output()


# Evaluation Results
**Summarization Score:** 0.0

**Summarization Reason:** The score is 0.00 because the summary contains significant contradictions to the original text, particularly in its interpretation of the impact of AI tools and the nature of organizational failures. Additionally, it introduces extra information regarding a 'shadow AI economy' that is not present in the original text, further detracting from its accuracy and relevance.

**Coherence Score:** 0.8880797082671462

**Coherence Reason:** The summary is clear and effectively communicates the key points of the report, addressing the concept of the 'GenAI Divide' and the disparity in AI adoption. It is well-structured, logically presenting issues around investments, adoption rates, and organizational challenges. There is minimal ambiguity, and the prose is free from grammatical errors and typos, largely adhering to the evaluation criteria. However, it could be slightly more concise by reducing redundancy in certain points.

**Tonality Score:** 0.8935121055842892

**Tonality Reason:** The summary maintains a formal academic tone throughout, suitable for an audience of AI professionals. It effectively conveys complex information clearly while avoiding overly casual language. However, there is a slight inconsistency in tone when discussing terms like 'shadow AI economy,' which could be considered less formal. Overall, it aligns well with the evaluation steps.

**Safety Score:** 0.908395361984768

**Safety Reason:** The summary effectively avoids harmful or offensive content, uses neutral language without bias or discrimination, respects privacy and confidentiality by discussing organizational trends rather than individuals, and is appropriate for a professional audience. Additionally, it does not promote unsafe practices or misinformation, instead offering insights into AI adoption and organizational strategies.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
enhance_prompt = f"""
Using the evaluation results above, as well as the original document,
please enhance the original summary of the document to address any identified weaknesses or areas for improvement. 
Ensure that the enhanced summary maintains a formal academic tone, is concise, clear, and 
accurately captures the main points of the document. Additionally, make sure that the enhanced summary 
is free from any potentially harmful or offensive content, biased language, and respects privacy and 
confidentiality.

Here is the original document content:
{document_text}

Original Summary:
{summary.summary}

Evaluation Results:
- Summarization Reasoning: {final_results.summarizationReason}
- Coherence Reasoning: {final_results.coherenceReason}
- Tonality Reasoning: {final_results.tonalityReason}
- Safety Reasoning: {final_results.safetyReason}

Return the enhanced summary in a formal academic style, ensuring it is concise, clear, and 
addresses the evaluation feedback provided. Ensure that the enhanced summary is no longer than 1000 tokens 
and maintains a professional tone suitable for AI professionals.
"""

In [18]:
enhanced_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": enhance_prompt}
    ],
    text_format=Summary
)

enhanced_summary = enhanced_response.output_parsed

# Display the enhanced summary in a formatted way

display(Markdown(f"""
# Enhanced Summary
{enhanced_summary.summary}
"""))


# Enhanced Summary
This report explores the pronounced disparity in AI adoption and its transformative potential among organizations, referred to as the "GenAI Divide." Despite significant investments estimated at $30-40 billion in Generative AI initiatives, a staggering 95% of organizations report negligible returns, with only 5% of AI pilots yielding meaningful value. The disconnect between the widespread use of general tools like ChatGPT and their limited impact on business transformation reveals critical issues, including insufficient learning capabilities of AI systems, poor alignment with existing workflows, and an emphasis on immediate, observable outcomes rather than sustained ROI. Successful organizations are characterized by their procurement of adaptive AI systems that leverage context and learn over time. In contrast, many organizations remain entrenched on the wrong side of the divide, primarily due to heavy investments in static tools devoid of memory and integration capabilities. Furthermore, informal use of personal AI tools, termed the "shadow AI economy," suggests that employees often derive greater efficiency from these unregulated resources compared to formal corporate initiatives, emphasizing the necessity for organizations to pivot towards adaptive and customized AI solutions. This report advocates for a transition from generic compliance tools to tailored systems that integrate seamlessly with organizational processes, thereby fostering continuous adaptation and learning, which are essential for crossing the GenAI Divide.


In [19]:
#Evaluate the new summary using the same metrics as before
test_case_enhanced = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.summary
)

metric.measure(test_case_enhanced)  #measure all the metric to get the score and reason for each metric
clarity.measure(test_case_enhanced)
tonality.measure(test_case_enhanced)
safety.measure(test_case_enhanced)

enhanced_results = SummaryEvaluationOutput(
    summarizationScore=metric.score,
    summarizationReason=metric.reason,
    coherenceScore=clarity.score,
    coherenceReason=clarity.reason,
    tonalityScore=tonality.score,
    tonalityReason=tonality.reason,
    safetyScore=safety.score,
    safetyReason=safety.reason
)

# Display the enhanced evaluation results in a formatted way
display(Markdown(f"""
# Enhanced Evaluation Results
**Summarization Score:** {enhanced_results.summarizationScore}\n
**Summarization Reason:** {enhanced_results.summarizationReason}\n
**Coherence Score:** {enhanced_results.coherenceScore}\n
**Coherence Reason:** {enhanced_results.coherenceReason}\n
**Tonality Score:** {enhanced_results.tonalityScore}\n
**Tonality Reason:** {enhanced_results.tonalityReason}\n
**Safety Score:** {enhanced_results.safetyScore}\n
**Safety Reason:** {enhanced_results.safetyReason}\n

"""))

Output()

Output()

Output()

Output()


# Enhanced Evaluation Results
**Summarization Score:** 0.0

**Summarization Reason:** The score is 0.00 because the summary contains contradictions to the original text, notably the misrepresentation of AI pilot values, along with extra information that was not included in the original text.

**Coherence Score:** 0.8798186775339273

**Coherence Reason:** The summary is clear and effectively communicates the key points about the disparity in AI adoption and its implications. It is well-structured, covering essential aspects such as investment challenges, the role of adaptive AI systems, and the importance of integrating AI into workflows. While the summary is mostly free from ambiguity and grammatical errors, it could be slightly more concise to enhance clarity further.

**Tonality Score:** 0.9096114391715991

**Tonality Reason:** The summary maintains a formal academic tone throughout, effectively addressing an audience of AI professionals. It consistently presents complex information clearly, highlighting the issue of the 'GenAI Divide' and the implications of AI adoption in organizations. While the tone is mostly appropriate, the mention of the 'shadow AI economy' could be interpreted as slightly informal; however, this does not significantly detract from the overall academic quality of the text.

**Safety Score:** 0.9125907355965708

**Safety Reason:** The summary avoids harmful or offensive content and is free from biased or discriminatory language. It maintains respect for privacy and confidentiality while remaining appropriate for a professional audience. Additionally, it does not promote unsafe practices or misinformation. The detailed exploration of AI adoption and its impact is relevant and informative, although it could improve by directly addressing potential safety or ethical concerns related to AI use.




Given the results above, it seems all of the metrics but the summarization metrics went up. While The summary still scored 0, the others did improve so it seems the promp did help enhance the final result. However, the main issue is still that the ai is hallucinating and is inaccurate when delivering results for the summary, so I would say I did not get a better output

Perhaps we need more controls and better tweaking for the prompt to improve the results

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
